## Writing simple neural network from scratch - will train and test using MNIST dataset

In [36]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# MNIST dataset
mnist = fetch_openml('mnist_784', version=1)
X = mnist.data.astype(np.float32)
y = mnist.target.astype(int)

# normalising inputs
X /= 255.0

# train, test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)  # (56000, 784)

# ensuring correct datatypes
X_train = np.array(X_train, dtype=np.float32)
X_test  = np.array(X_test, dtype=np.float32)
y_train = np.array(y_train, dtype=int)
y_test  = np.array(y_test, dtype=int)

(56000, 784)


Softmax function: $ Softmax(x_i) = \frac{e^{x_i}}{\sum e^{x_j}} $.


In [19]:
# hidden layer(s) activation function
def relu(x):
    return np.maximum(0, x)

# final output layer activation function
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # we subtract from the max since it ensures stability (at no cost)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

Cross entropy loss function: $L = -\sum_{c=1}^{M} y_{o,c} \log(p_{o,c})$ 

where $ M $= total number of classes, 

$y_{o,c}$ is truth value (actual class) - one hot encoded, 

$p_{o,c}$ is the predicted probability that observation $o$ belongs to class $c$ (softmax function output)



In [18]:
# loss function
def cross_entropy_loss(y_pred, y_true):

    n = y_pred.shape[0]
    log_likelihood = -np.log(y_pred[range(n), y_true] + 1e-12) # add small value to prevent log(0), ensure numerical stability
    return np.mean(log_likelihood)

In the neural network, we will impliment one hidden layer - relu (and one output layer - softmax)

For computing backpropagation step, partial derivative of loss with respect to softmax input is $\frac{\partial L}{\partial z_2} = \text{Predictions} - \text{Targets}$



In [29]:
class NeuralNetwork:
    
    # pass in the input, hidden layer, output sizes, learning rate
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):

        self.lr = lr
        
        # random initialisation of weights and biases with He initialization (for ReLU)
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
        self.b1 = np.zeros((1, hidden_size))
        
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size)
        self.b2 = np.zeros((1, output_size))


    def forward(self, X):

        # computing dot product between inputs and weights, then adding bias, then applying activation function
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = relu(self.z1)
        
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)
        
        return self.a2

    def backward(self, X, y):

        n = X.shape[0]
        
        grad_z2 = self.a2.copy()
        grad_z2[np.arange(n), y] -= 1
        grad_z2 /= n
        
        # softmax layer gradients
        grad_W2 = np.dot(self.a1.T, grad_z2)
        grad_b2 = np.sum(grad_z2, axis=0, keepdims=True)
        
        grad_a1 = np.dot(grad_z2, self.W2.T)
        grad_z1 = grad_a1 * (self.z1 > 0)
        
        # relu layer gradients
        grad_W1 = np.dot(X.T, grad_z1)
        grad_b1 = np.sum(grad_z1, axis=0, keepdims=True)
        
        # subtracting lr from gradients to update weights and biases
        self.W2 -= self.lr * grad_W2
        self.b2 -= self.lr * grad_b2
        self.W1 -= self.lr * grad_W1
        self.b1 -= self.lr * grad_b1

    def train(self, X, y, epochs=10, batch_size=64):
        
        n = X.shape[0]
        
        for epoch in range(epochs):

            # shuffling order of data for each epoch
            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            

            # stochastic gradient descent in batches
            for i in range(0, n, batch_size):

                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                
                self.forward(X_batch)
                self.backward(X_batch, y_batch)
            
            # quick loss check
            probs = self.forward(X[:1000])
            loss = cross_entropy_loss(probs, y[:1000])
            
            print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


    def predict(self, X):
        
        probs = self.forward(X)
        return np.argmax(probs, axis=1) # returning the index of the highest probability as the predicted class

In [59]:
model = NeuralNetwork(input_size=784, hidden_size=128, output_size=10, lr=0.01)
model.train(X_train, y_train, epochs=15, batch_size=64)

Epoch 1, Loss: 0.4422
Epoch 2, Loss: 0.3131
Epoch 3, Loss: 0.2707
Epoch 4, Loss: 0.2405
Epoch 5, Loss: 0.2222
Epoch 6, Loss: 0.2091
Epoch 7, Loss: 0.1959
Epoch 8, Loss: 0.1842
Epoch 9, Loss: 0.1765
Epoch 10, Loss: 0.1715
Epoch 11, Loss: 0.1609
Epoch 12, Loss: 0.1531
Epoch 13, Loss: 0.1479
Epoch 14, Loss: 0.1424
Epoch 15, Loss: 0.1358


In [60]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1343
           1       0.95      0.98      0.97      1600
           2       0.95      0.93      0.94      1380
           3       0.93      0.93      0.93      1433
           4       0.93      0.94      0.94      1295
           5       0.94      0.93      0.93      1273
           6       0.96      0.96      0.96      1396
           7       0.95      0.95      0.95      1503
           8       0.93      0.92      0.93      1357
           9       0.93      0.92      0.93      1420

    accuracy                           0.94     14000
   macro avg       0.94      0.94      0.94     14000
weighted avg       0.94      0.94      0.94     14000



We now make a neural network with multiple hidden layers

In [43]:
class DeepNeuralNetwork:
    
    # two hidden relu layers, softmax output layer
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size, lr=0.01):

        self.lr = lr
        
        self.W1 = np.random.randn(input_size, hidden1_size) * np.sqrt(2. / input_size)
        self.b1 = np.zeros((1, hidden1_size))
        
        self.W2 = np.random.randn(hidden1_size, hidden2_size) * np.sqrt(2. / hidden1_size)
        self.b2 = np.zeros((1, hidden2_size))

        self.W3 = np.random.randn(hidden2_size, output_size) * np.sqrt(2. / hidden2_size)
        self.b3 = np.zeros((1, output_size))


    def forward(self, X):

        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = relu(self.z1)
    
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = relu(self.z2)
        
        self.z3 = np.dot(self.a2, self.W3) + self.b3
        self.a3 = softmax(self.z3)
        
        return self.a3 # returning final output


    def backward(self, X, y):
        n = X.shape[0]
        
        grad_z3 = self.a3.copy()
        grad_z3[np.arange(n), y] -= 1
        grad_z3 /= n
        
        grad_W3 = np.dot(self.a2.T, grad_z3)
        grad_b3 = np.sum(grad_z3, axis=0, keepdims=True)
        
        grad_a2 = np.dot(grad_z3, self.W3.T)
        grad_z2 = grad_a2 * (self.z2 > 0)
        
        grad_W2 = np.dot(self.a1.T, grad_z2)
        grad_b2 = np.sum(grad_z2, axis=0, keepdims=True)

        grad_a1 = np.dot(grad_z2, self.W2.T)
        grad_z1 = grad_a1 * (self.z1 > 0)
        
        grad_W1 = np.dot(X.T, grad_z1)
        grad_b1 = np.sum(grad_z1, axis=0, keepdims=True)
        

        # update weights and biases
        self.W3 -= self.lr * grad_W3
        self.b3 -= self.lr * grad_b3
        self.W2 -= self.lr * grad_W2
        self.b2 -= self.lr * grad_b2
        self.W1 -= self.lr * grad_W1
        self.b1 -= self.lr * grad_b1


    def train(self, X, y, epochs=10, batch_size=64):

        n = X.shape[0]

        for epoch in range(epochs):

            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            for i in range(0, n, batch_size):

                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                
                self.forward(X_batch)
                self.backward(X_batch, y_batch)
            
            # loss
            probs = self.forward(X[:1000])
            loss = cross_entropy_loss(probs, y[:1000])
            print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)

In [55]:
model2 = DeepNeuralNetwork(input_size=784, hidden1_size=256, hidden2_size=128, output_size=10, lr=0.01)
model2.train(X_train, y_train, epochs=15, batch_size=64)

Epoch 1, Loss: 0.3279
Epoch 2, Loss: 0.2440
Epoch 3, Loss: 0.1974
Epoch 4, Loss: 0.1706
Epoch 5, Loss: 0.1550
Epoch 6, Loss: 0.1349
Epoch 7, Loss: 0.1231
Epoch 8, Loss: 0.1132
Epoch 9, Loss: 0.1064
Epoch 10, Loss: 0.0994
Epoch 11, Loss: 0.0937
Epoch 12, Loss: 0.0896
Epoch 13, Loss: 0.0812
Epoch 14, Loss: 0.0788
Epoch 15, Loss: 0.0734


In [56]:
y_pred2 = model2.predict(X_test)
print(classification_report(y_test, y_pred2))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1343
           1       0.97      0.98      0.98      1600
           2       0.96      0.96      0.96      1380
           3       0.96      0.94      0.95      1433
           4       0.95      0.96      0.95      1295
           5       0.95      0.95      0.95      1273
           6       0.96      0.98      0.97      1396
           7       0.96      0.97      0.97      1503
           8       0.95      0.94      0.95      1357
           9       0.95      0.94      0.95      1420

    accuracy                           0.96     14000
   macro avg       0.96      0.96      0.96     14000
weighted avg       0.96      0.96      0.96     14000



In [64]:
print("One relu layer: ", accuracy_score(y_test, y_pred))
print("Two relu layers: ", accuracy_score(y_test, y_pred2))


One relu layer:  0.9445
Two relu layers:  0.9598571428571429
